# MLS211 Report Data Reader

## Task 1a

In [ ]:
# Import all necessary libraries

# Download mls211 library if not already available
import os
import pathlib
import urllib.request

url = "https://raw.githubusercontent.com/dche658/mls211/refs/heads/main/mls211.py"
output_path = "mls211.py"

# Check if the file already exists
if not os.path.exists(output_path):
    print(f"Downloading {url}...")
    urllib.request.urlretrieve(url, output_path)
    print("Download complete.")
else:
    print(f"File '{output_path}' already exists. Skipping download.")

# Remainder of imports here
from mls211 import (
    ExcelWriter,
    StandardCurveAnalyser,
    StandardCurveBuilder,
    round_half_up,
)




File 'mls211.py' already exists. Skipping download.


In [2]:
# Read the copied task 1a data

DEFAULT_EXCEL_FILE = "./Task 1a/Task 1a.xlsx"

# S_Mean is the mean value submitted by the student
# S_Blanked is the blanked value submitted by the student
colnames = ["ID","Conc","Abs1", "Abs2", "S_Mean", "S_Blanked"]

# Copy submitted results to here
input_text = """
B 0.0 0.117 0.099 0.108 0.0
S1 0.1 0.245 0.263 0.254 0.146
S2 0.2 0.448 0.443 0.445 0.337
S3 0.5 1.020 1.045 1.032 0.924
S4 0.8 1.590 1.613 1.613 1.505
S5 1.0 2.013 2.036 2.024 1.916
"""

builder = StandardCurveBuilder()
df = builder.read(input_text.strip()).set_colnames(colnames).calc_abs_means(["Abs1", "Abs2"]).build()
analyser = StandardCurveAnalyser(df)
reg = analyser.linear_regression("Conc","Blanked_Abs")
r_squared = analyser.r_squared_forced_through_origin("Conc", "Blanked_Abs")

# Calculate concentration based on given absorbance
da_abs = 0.752
da_dilution = 5
da_conc = da_dilution*da_abs/reg[0]
# Round mean and blanked absorbances just prior to printing
# Round exactly to three decimal places
df['Mean_Abs'] = round_half_up(df['Mean_Abs'], "0.001")
df['Blanked_Abs'] = round_half_up(df['Blanked_Abs'], "0.001")
# Print output
print(df)
print(f"Slope: {reg[0]:.4f}, Intercept: {reg[1]:.4f}")
print(f"R-squared: {r_squared:.4f}")
print(f"Conc of unknown (abs={da_abs:.3f}): {da_conc:.1f} mmol/L")


   ID  Conc   Abs1   Abs2  S_Mean  S_Blanked Mean_Abs Blanked_Abs
0   B   0.0  0.117  0.099   0.108      0.000    0.108       0.000
1  S1   0.1  0.245  0.263   0.254      0.146    0.254       0.146
2  S2   0.2  0.448  0.443   0.445      0.337    0.446       0.338
3  S3   0.5  1.020  1.045   1.032      0.924    1.033       0.925
4  S4   0.8  1.590  1.613   1.613      1.505    1.602       1.494
5  S5   1.0  2.013  2.036   2.024      1.916    2.024       1.916
Slope: 1.8844, Intercept: 0.0000
R-squared: 0.9993
Conc of unknown (abs=0.752): 2.0 mmol/L


In [3]:
# Write to Excel
writer = ExcelWriter()
excel_output_path = DEFAULT_EXCEL_FILE
raw_data = df.iloc[:, [2,3]]
writer.write_dataframe_to_excel(raw_data, pathlib.Path(excel_output_path), "Table 3", 3, 6)
print(f"Data written to Excel file: {excel_output_path}")

Data written to Excel file: ./Task 1a/Task 1a.xlsx
